# Day 19: Function Calling

Walkthrough of structured tool calls: JSON schemas, a scripted fake model, the function calling loop, argument validation, parallel calls, and a comparison with Day 17's ReAct text format. Everything here runs offline with no API key and no vendor SDK.

## 1. What is function calling

Instead of parsing free text like Day 17's ReAct format, the model returns a structured tool call: a name and JSON arguments, already parsed and typed.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
sys.path.insert(0, os.path.abspath('../day-15-agentic-ai'))
sys.path.insert(0, os.path.abspath('../day-16-custom-tools'))
sys.path.insert(0, os.path.abspath('../day-17-react-pattern'))
print("Free text has to be parsed. A structured tool call does not.")
print("Loop: send messages and tool schemas, get a tool call or a final answer, run the tool, repeat.")

## 2. Build schemas from functions

Using inspect to turn a Day 16 tool function into a JSON schema: name, description, required fields, and inferred types.

In [ ]:
from tool_schemas import build_json_schema
from support_tools import lookup_order, create_ticket

schema = build_json_schema(lookup_order)
print(schema)
print()
print(build_json_schema(create_ticket))

## 3. Fake model response shape

A scripted model that returns the same response shape a real function calling API would: role, content, tool_calls, stop_reason.

In [ ]:
from fake_function_calling_model import fake_model_respond

messages = [{"role": "user", "content": "Is order ORD-1003 eligible for a refund?"}]
response = fake_model_respond(messages, tool_schemas=[])
print(response)

## 4. Run the function calling loop

Sending messages and tool schemas, running each tool call, appending the result tagged with its id, until the model answers in plain text.

In [ ]:
from function_calling_loop import run_function_calling_loop

answer = run_function_calling_loop("Is order ORD-1003 eligible for a refund?", tool_schemas=[])
print(f"\nFinal answer: {answer}")

## 5. Validate arguments

Checking a parsed argument dict against its schema before the tool runs: missing required fields, wrong types, and unexpected extra fields all get caught with a clear error.

In [ ]:
from argument_validation import validate_arguments

schema = build_json_schema(create_ticket)
for label, arguments in [
    ("valid", {"issue": "Item arrived damaged", "priority": "high"}),
    ("missing field", {"priority": "high"}),
    ("wrong type", {"issue": 12345}),
]:
    is_valid, error = validate_arguments(schema, arguments)
    print(f"{label}: valid={is_valid}" + (f", {error}" if error else ""))

## 6. Parallel tool calls

One model turn can request several independent tool calls at once, each tagged with its own id so results can be matched back correctly.

In [ ]:
from parallel_tool_calls import fake_model_parallel_response, run_parallel_tool_calls

response = fake_model_parallel_response()
print(f"Tool calls requested in this turn: {len(response['tool_calls'])}\n")
results = run_parallel_tool_calls(response)

## 7. Function calling vs ReAct

The same task shown in both formats, with a few simulated malformed outputs for each, to compare how each one fails.

In [ ]:
from function_calling_vs_react import (
    MALFORMED_REACT_EXAMPLES, MALFORMED_FUNCTION_CALL_EXAMPLES,
    try_parse_react, try_parse_function_call,
)

react_failures = sum(1 for text in MALFORMED_REACT_EXAMPLES if not try_parse_react(text)[0])
fc_failures = sum(1 for tc in MALFORMED_FUNCTION_CALL_EXAMPLES if not try_parse_function_call(tc, {"lookup_order"})[0])

print(f"ReAct text: {react_failures} of {len(MALFORMED_REACT_EXAMPLES)} malformed examples failed to parse")
print(f"Function calling: {fc_failures} of {len(MALFORMED_FUNCTION_CALL_EXAMPLES)} malformed examples failed to parse")

## 8. Key takeaways

- Function calling replaces free text parsing with a structured tool call: a name and JSON arguments the model returns directly.
- JSON schemas built from a function's real signature and docstring can never drift from what the function actually accepts, the same guarantee Day 16's tool_builder.py gave the Day 15 ToolRegistry.
- The function calling loop is the same think, decide, act, observe shape as Day 15 and Day 17, just with a structured tool call standing in for parsed text.
- Validating arguments against the schema before a tool runs catches a malformed call with a specific, actionable error.
- A single model turn can request several independent tool calls at once, matched back to their results by a shared id, not by the order they happen to arrive in.
- Free text has more ways to go wrong than structured data does: the same task's malformed examples showed three unrelated ReAct failures against two well-defined, mechanically checkable function calling failures.
- None of this is connected to a real API in this project. Every example here runs with a scripted fake model, and real_api_reference.py shows the real shapes as text only, never as executable code against a live key.